# 1. RAG 

The minsearch library will give us the tools to create a knowledge base and then use it to answer questions. This KB will be stored in memory, so once the session is finished, the KB will not be stored.

We will load the libraries and then define a RAG in 3 steps:
1. Search: Take the question and use the index to find the best responses. 
2. Build Prompt: Use the search results and the build_context results to build the prompt we're going to pass to the LLM.
3. LLM: Pass onto the model the Prompt with the best-matching results of the search and retrieve the answer.

In [1]:
# Library to read .env file
from dotenv import load_dotenv
load_dotenv()

# Library to connect to OpenAI, using the API key in the .env file
from openai import OpenAI
openai_client = OpenAI()

In [2]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw: 
    course_url = f'{url_prefix}/{course["path"]}'
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

# Test one of the questions from the Q&A.
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [3]:
# Use the search engine to find the relevant documents.

from minsearch import Index

index = Index(
    text_fields = ['question', 'section', 'answer'],
    # keyword_fields require an exact match, it is usually used to reduce to a subset.
    keyword_fields = ['course']
)

index.fit(documents)

index.search('Can I join the LLM Zoomcamp today?')

[{'id': '930286278d',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Where can I find previous LLM Zoomcamp projects?',
  'answer': 'You can browse previous LLM Zoomcamp project submissions here:\n\n- [2024 projects](https://courses.datatalks.club/llm-zoomcamp-2024/projects)\n- [2025 projects](https://courses.datatalks.club/llm-zoomcamp-2025/projects)\n\nThese pages show submitted repositories and can help you understand the expected scope and quality of capstone projects.'},
 {'id': '20c5a1347e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?',
  'answer': 'Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nIt contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps.'},
 {'id': 'fde155ddfb',
  '

In [4]:
# Create the search function.

def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict={'course': course}

    search_results = index.search(
        question,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=5
    )
    return search_results

In [5]:
# Test the search function.

search_results = search('Can I join the LLM Zoomcamp today?')
search_results

[{'id': '930286278d',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Where can I find previous LLM Zoomcamp projects?',
  'answer': 'You can browse previous LLM Zoomcamp project submissions here:\n\n- [2024 projects](https://courses.datatalks.club/llm-zoomcamp-2024/projects)\n- [2025 projects](https://courses.datatalks.club/llm-zoomcamp-2025/projects)\n\nThese pages show submitted repositories and can help you understand the expected scope and quality of capstone projects.'},
 {'id': '20c5a1347e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?',
  'answer': 'Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nIt contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps.'},
 {'id': '054f3fd58f',
  '

In [6]:
# Function to build the prompt.
instructions = '''
You are a helpful assistant that can answer questions about the DataTalksClub courses.
Use the context to find the relevant information and answer the question.
'''

user_prompt_template = '''
Question: {question}
Context: {context}
'''

In [7]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')
    
    return '\n'.join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = user_prompt_template.format(
        question=question,
        context=context
    )
    return prompt

question = 'Can I still join the LLM Zoomcamp course?'
prompt = build_prompt(question, search_results)
print(prompt)


Question: Can I still join the LLM Zoomcamp course?
Context: Capstone Project
Q: Where can I find previous LLM Zoomcamp projects?
A: You can browse previous LLM Zoomcamp project submissions here:

- [2024 projects](https://courses.datatalks.club/llm-zoomcamp-2024/projects)
- [2025 projects](https://courses.datatalks.club/llm-zoomcamp-2025/projects)

These pages show submitted repositories and can help you understand the expected scope and quality of capstone projects.

General Course-Related Questions
Q: Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?
A: Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

It contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps.

General Course-Related Questions
Q: Where is the LLM Zoomcamp Telegram channel?
A: The Telegram channel is [https://t.me/llm_zoomcamp](https://t.me/llm_zo

In [ ]:
def llm(instructions = instructions, user_prompt = prompt, model = 'gpt-5.4-mini'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create( 
        model = model,
        input = message_history
    )

    # Show the usage of the LLM in tokens.
    print(f'Usage: Input Tokens = {response.usage.input_tokens}. Output Tokens = {response.usage.output_tokens}.\n----')
    return response.output_text

def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(instructions, prompt, model = model)
    return answer

In [14]:
question = 'I just discovered the course. Can I join now?'
answer = rag(question)
print(answer)

Usage: Input Tokens = 516. Output Tokens = 37.
----

Yes — you can still join and start learning now.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [15]:
question = 'Will I get a certificate?'
answer = rag(question)
print(answer)

Usage: Input Tokens = 344. Output Tokens = 69.
----

Yes — but only if you finish the course with a **live cohort** and complete the **Capstone project**.

You **won’t** get a certificate in **self-paced mode**. Also, you need to **peer-review 3 capstone projects** during the course run, since certificates depend on that process.
